# CS3264 Tutorial 09 — Problem 3
## Core exercise: one fit

1. Write the sampled ELBO in part (a).
2. Fill in the **three expressions** in `sampled_elbo` below.
3. Run the supplied check and fit once with **K = 10**.
4. Inspect the two plots and discuss part (c).

The gradients, optimiser, numerical reference and plotting are supplied. No numerical integration code, gradient derivation, repeated runs or results table is required. Use Python 3 with NumPy and Matplotlib.

We choose the Laplace family $\mathcal Q=\{q_{\mu,a}:\mu,a\in\mathbb R\}$, where $q_{\mu,a}(z)=\operatorname{Laplace}(z;\mu,e^a)$, and optimise its parameters $(\mu,a)$. We sample from the current member $q_{\mu,a}\in\mathcal Q$. These are not samples from the exact posterior.


### 1. Provided setup — run as is

The model has a standard normal prior and 8 successes in 10 trials. `log_likelihood(z)` evaluates the full binomial log likelihood, with softplus calculated stably. The reference is obtained by one-dimensional numerical integration. Support code is collapsed where your notebook viewer supports it.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

x, n = 8, 10
log_choose = math.lgamma(n+1)-math.lgamma(x+1)-math.lgamma(n-x+1)
log_2pi = math.log(2*math.pi)
c0 = 0.5*log_2pi - 1 - math.log(2)
integrate = getattr(np, 'trapezoid', None) or np.trapz

def sigmoid(z):
    return np.exp(-np.logaddexp(0.0, -np.asarray(z)))

def log_likelihood(z):
    return log_choose + x*z - n*np.logaddexp(0.0, z)

def log_normal(z, mu=0.0, s=1.0):
    return -0.5*log_2pi - np.log(s) - 0.5*((z-mu)/s)**2

def reference(grid):
    log_joint = log_likelihood(grid)+log_normal(grid)
    shift = np.max(log_joint)
    unnormalised = np.exp(log_joint-shift)
    normaliser = integrate(unnormalised,grid)
    return unnormalised/normaliser, float(np.log(normaliser)+shift)

grid=np.linspace(-8,8,16001)
posterior, log_evidence=reference(grid)
# Provided check on truncation and integration error.
wide_grid=np.linspace(-10,10,40001)
wide_posterior, wide_log_evidence=reference(wide_grid)
assert abs(wide_log_evidence-log_evidence)<1e-7


### 2. Your task — complete three expressions

Here `a` is the **log scale**, so `b = exp(a)` and the variance is `2*b**2`. `eps` contains noise from `Laplace(0,1)`, already drawn by the caller. Use those values; do not draw fresh noise inside this function.

Replace the three `...` placeholders. Return the sampled likelihood term **minus** the analytic prior KL. Keep the constants already included in `log_likelihood`. The supplied constant is $c_0=\tfrac12\log(2\pi)-1-\log 2$, and the prior KL is $\mu^2/2+e^{2a}-a+c_0$.


In [ ]:
def sampled_elbo(mu, a, eps):
    z = ...                 # TODO: reparameterised samples from q
    likelihood_term = ...   # TODO: average log_likelihood over these samples
    prior_kl = ...          # TODO: analytic KL from q to N(0,1)
    if any(term is Ellipsis for term in (z, likelihood_term, prior_kl)):
        raise NotImplementedError('Replace the three ... placeholders above.')
    return float(likelihood_term - prior_kl)


### 3. Provided check — run as is

The supplied gradients should agree with derivatives of your objective. Both finite-difference evaluations use the same noise. This is a check of your three expressions, not an additional derivation exercise.


In [ ]:
def provided_gradient(mu,a,eps):
    b=np.exp(a)
    r=x-n*sigmoid(mu+b*eps)
    return np.array([np.mean(r)-mu, np.mean(r*b*eps)+1-2*b*b])

eps_check=np.random.default_rng(123).laplace(size=50)
# At mu = a = 0, KL[Laplace(0,1) || N(0,1)] = 1 + c0.
np.testing.assert_allclose(sampled_elbo(0.0,0.0,eps_check),
                           np.mean(log_likelihood(eps_check))-(1+c0),atol=1e-10)
for point in (np.array([0.,0.]),np.array([0.5,-0.2]),np.array([1.2,-0.8])):
    h=1e-5
    numeric=np.zeros(2)
    for j in range(2):
        offset=np.eye(2)[j]*h
        numeric[j]=(sampled_elbo(*(point+offset),eps_check)
                    -sampled_elbo(*(point-offset),eps_check))/(2*h)
    np.testing.assert_allclose(numeric,provided_gradient(*point,eps_check),
                               atol=1e-7,rtol=1e-6)
print('Your sampled ELBO passes the derivative check.')


### 4. One fit and two plots — run as is

The supplied optimiser starts from `mu = 0, a = 0`, takes 5000 gradient-ascent steps with fresh samples, and uses `K = 10`, seed `0`. The step size is `0.03 / (1 + t/500)`. Only the Laplace parameters change.

The left plot is a **sampled** ELBO trace. The right plot compares the fitted Laplace density with the numerical posterior. Read the automatically generated output; no table or hand calculation is needed.


In [ ]:
K=10
rng=np.random.default_rng(0)
parameters=np.array([0.0,0.0])
history=np.empty(5000)
for t in range(5000):
    mu,a=parameters
    eps=rng.laplace(size=K)
    history[t]=sampled_elbo(mu,a,eps)
    parameters += (0.03/(1+t/500))*provided_gradient(mu,a,eps)
    if not np.all(np.isfinite(parameters)):
        raise FloatingPointError('Non-finite parameters.')
mu,a=parameters
b=float(np.exp(a))
q=np.exp(-np.abs(grid-mu)/b)/(2*b)
kl_prior=0.5*mu*mu+b*b-a+c0
# Provided quadrature: average over the two exponential halves of Laplace noise.
def integrated_likelihood(order):
    nodes,weights=np.polynomial.laguerre.laggauss(order)
    return 0.5*np.dot(weights,log_likelihood(mu+b*nodes)+log_likelihood(mu-b*nodes))
integrated_elbo=float(integrated_likelihood(64)-kl_prior)
checked_elbo=float(integrated_likelihood(128)-kl_prior)
assert abs(integrated_elbo-checked_elbo)<1e-7
print(f'Fitted Laplace: mu = {mu:.5f}, scale b = {b:.5f}')
print(f'Numerically integrated ELBO: {integrated_elbo:.6f}')
print(f'Numerical log evidence: {log_evidence:.6f}')
print(f'ELBO gap: {log_evidence-integrated_elbo:.6f}')

fig,axes=plt.subplots(1,2,figsize=(11,3.6))
axes[0].plot(history,color='#307A96',linewidth=0.65,alpha=0.6)
axes[0].axhline(log_evidence,color='#222222',linestyle='--',linewidth=1,
                label='Numerical log evidence')
axes[0].set(xlabel='Update',ylabel='Sampled ELBO',title='Noisy objective estimates')
axes[0].legend(frameon=False,fontsize=8)
axes[1].plot(grid,posterior,color='#222222',linewidth=2.2,label='Numerical posterior')
axes[1].plot(grid,q,color='#258168',linestyle='--',linewidth=1.7,label='Fitted Laplace q')
axes[1].set(xlim=(-1.5,3.5),xlabel='Latent log odds z',ylabel='Density',title='The fitted approximation')
axes[1].legend(frameon=False,fontsize=8)
for ax in axes: ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()


### 5. Discuss — no further code needed

- Must every update increase the true ELBO? Is the trace showing the true ELBO?
- Does increasing $K$ change which posterior shapes $\mathcal Q$ can represent?
- Could a sampled ELBO estimate exceed log evidence? Which quantity has the lower-bound guarantee?
